# Hands-On AI for Science
## August 12, Afternoon: Your Data as a Matrix

### Self-contained Colab notebook

Everything you need is in this notebook — there is nothing to install and nothing to download. Just run the cells in order.


Every method you will see for the rest of this workshop — regression, neural networks, embeddings, foundation models — uses an identical data structure:

* a matrix $X$ with one **row per sample/datapoint** and one **column per feature** (shape $n \times d$), and
* a vector $y$ of **labels**, one per sample (length $n$).

Your data does not start out looking like that.  It (probably) starts as a spreadsheet or a csv file, possibly quite complicated. Some columns are text, some values are missing, someone typed "Control" three different ways.  This hour is about getting from that spreadsheet to $(X, y)$.  Everything else this week assumes you can do this.

## Setup

Before anything else, the cell below checks that this notebook has what it needs: the required packages, and the data/homework files that ship with the workshop repository.  **If anything prints MISSING, fix it now, before going further.**

* A MISSING *package* under Anaconda usually means Anaconda isn't the Python running this notebook — ask an instructor.
* A MISSING *file* usually means Jupyter was started in the wrong folder.  Everything in this workshop assumes the notebook is opened from the repository's `lectures` folder (see the first notebook, `a12am_python.ipynb`, for how the repository is downloaded).
* After installing anything, always **restart the kernel** (Kernel → Restart Kernel) and re-run from the top.

### Data files

Run this cell to create the data files this notebook reads. You do not need to edit it.

In [ ]:
# The data files for this notebook, embedded so nothing needs downloading.
# Run this cell once; it recreates the .csv files next to the notebook.
import base64, pathlib

_DATA = {
    'a12pm_measurements.csv':
        'c2FtcGxlX2lkLG1vdXNlX2lkLGNvbmRpdGlvbixnZW5lX2EsZ2VuZV9iLGdlbmVfYyxnZW5lX2QKczAwMSxtMDAxLGNvbnRyb2wgLDQuNTMsMi4xLDEwLjMxNCw1LjU0OApzMDAyLG0wMDEsQ29udHJvbCwzLjY1NCwyLjE3MiwxNC45MDcsNi4xMQpzMDAzLG0wMDEsY29udHJvbCwzLjg2OCwzLjI1NSw4LjgzOCw1Ljk1MgpzMDA0LG0wMDEsY29udHJvbCAsNy4zNCwyLjUwMiwxNC41Niw3LjY4MwpzMDA1LG0wMDEsY29udHJvbCw0LjQzMyw0LjM3MiwxMS45NjUsOS4yOTEKczAwNixtMDAyLENPTlRST0wsNC44NDIsMy4yNSwxMS43MDYsNS4xNDkKczAwNyxtMDAyLGNvbnRyb2wsNC41ODgsMS45MzEsLDcuODgyCnMwMDgsbTAwMixjb250cm9sICwyLjgyNywwLjk3OSwsNy4xNzYKczAwOSxtMDAyLGNvbnRyb2wgLDQuNDAyLDMuMDM1LDguODM0LDUuNDMKczAxMCxtMDAyLENPTlRST0wsMi44MjcsMi43NTYsMTIuODU0LDYuNDY1CnMwMTEsbTAwMyxDT05UUk9MLDUuMzgzLDMuOTM2LDEzLjIxNSw4LjcyMgpzMDEyLG0wMDMsQ09OVFJPTCw0LjYxNywxLjEzMSwxMi40NjksNi4wODIKczAxMyxtMDAzLGNvbnRyb2wgLDcuMDgzLDIuMzE2LDEzLjc3LDguNDE5CnMwMTQsbTAwMyxjb250cm9sICw1Ljg4LDQuNTM3LDExLjMyMyw1Ljk3OApzMDE1LG0wMDMsY29udHJvbCw2LjEzOSwsMTIuNjY1LDkuNDM1CnMwMTYsbTAwNCxjb250cm9sICwzLjg1NSwsOS45NTcsOC4yMjgKczAxNyxtMDA0LENPTlRST0wsNC40NjMsNS4wMjIsMTMuODU0LDkuNjg2CnMwMTgsbTAwNCxDT05UUk9MLCwwLjQ1NSwxNS4xOTMsNC44NzUKczAxOSxtMDA0LGNvbnRyb2wgLDUuMTMyLDEuOTY2LDE1LjI2NSw5LjMyNApzMDIwLG0wMDQsQ29udHJvbCwsMi44ODYsMTIuNTQzLDUuMzU2CnMwMjEsbTAwNSxDT05UUk9MLDQuMDc2LDMuNzc3LDkuNDY3LDYuMzIKczAyMixtMDA1LGNvbnRyb2wgLDIuNjc1LDMuNDI1LDEzLjgwNSw3LjkzMgpzMDIzLG0wMDUsQ09OVFJPTCw1Ljg2OCwxLjk4NiwxNy42NDcsNS42MgpzMDI0LG0wMDUsY29udHJvbCwsMy42NzEsMTAuNTYxLDkuMDkzCnMwMjUsbTAwNSxDb250cm9sLDUuNDk5LDIuMjUzLDE0LjkyNSw1LjcwOQpzMDI2LG0wMDYsY29udHJvbCw1Ljg5MSwyLjA0LDEyLjA4Nyw5Ljg4NApzMDI3LG0wMDYsQ09OVFJPTCw1LjA5OSwzLjUzNiwxMC41NDUsOC4xMwpzMDI4LG0wMDYsY29udHJvbCw1LjU3NiwsMTEuNTkyLDEuMzc5CnMwMjksbTAwNixjb250cm9sICw0LjU4MSwyLjA5NywxMy41NTYsNy4zOTkKczAzMCxtMDA2LGNvbnRyb2wgLDQuNjcsMC45ODMsOS4wODEsNy41NTYKczAzMSxtMDA3LENvbnRyb2wsNC40NjksMy4xODksMTIuMzQzLDcuMDU0CnMwMzIsbTAwNyxjb250cm9sLDUuMTgxLDMuMjc5LCw2LjIyOApzMDMzLG0wMDcsY29udHJvbCw1LjY1OSwzLjEyOCwxMC41Nyw3LjU1NApzMDM0LG0wMDcsQ29udHJvbCw1Ljg4NywxLjk0OCwxMS42NDQsNy4wOTcKczAzNSxtMDA3LGNvbnRyb2wsNC40MDQsMS40NzIsOS4zMTQsNC42OTQKczAzNixtMDA4LGNvbnRyb2wsNC4yNTUsMi4yNzQsMTIuNDI0LDYuNzE5CnMwMzcsbTAwOCxjb250cm9sLDQuMDkzLCwxMS4yODgsNi42NzkKczAzOCxtMDA4LGNvbnRyb2wgLDYuNDM4LDQuMzQ5LDE0LjYzNiwxMC4xMTkKczAzOSxtMDA4LENPTlRST0wsNS40ODgsMy41NzYsMTIuNjMyLDEwLjE5CnMwNDAsbTAwOCxjb250cm9sICw3LjM1MywzLjc5NCwxMi45ODYsNS4zMTYKczA0MSxtMDA5LGNvbnRyb2wsNC41ODgsMS4zMzMsMTAuNzEyLDcuNTQKczA0MixtMDA5LGNvbnRyb2wgLDUuNiwyLjU1NiwxMC40MjIsNy4wMDkKczA0MyxtMDA5LGNvbnRyb2wsMy41MDEsMy40OTcsMTUuMzQxLDguMjIxCnMwNDQsbTAwOSxDT05UUk9MLCwyLjc1OCw3LjQyOSw0LjI1NgpzMDQ1LG0wMDksY29udHJvbCAsNC43NDIsMS45NzcsNi43ODUsNS4wOTcKczA0NixtMDEwLENvbnRyb2wsNS4wODEsMy43MzgsMTQuMjQ3LDYuNDkxCnMwNDcsbTAxMCxDT05UUk9MLDQuOTExLDIuNTUyLDEwLjcxLDguNDQxCnMwNDgsbTAxMCxjb250cm9sICw0LjAyLDQuMTksMTQuMTU1LDcuMDU0CnMwNDksbTAxMCxjb250cm9sLDMuNjM3LDUuNDA5LDEzLjE1OCw3LjI1OApzMDUwLG0wMTAsQ09OVFJPTCw1LjEwNiwzLjUsOS44OTgsNy40MjUKczA1MSxtMDExLGNvbnRyb2wsNC4wNDUsMy42MTgsMTEuMTk5LDcuNDAzCnMwNTIsbTAxMSxjb250cm9sLDcuNjg0LDMuNTI3LDExLjk3NixuLmQuCnMwNTMsbTAxMSxjb250cm9sLDUuODA3LDQuMzI1LDExLjU2OSw5LjUxNQpzMDU0LG0wMTEsY29udHJvbCw0LjYxMSwwLjk1LDEwLjEzNCw1LjQxNQpzMDU1LG0wMTEsY29udHJvbCw1LjM1LDEuNzk0LDExLjI4LDcuMjk5CnMwNTYsbTAxMixjb250cm9sICwzLjg3MSwyLjg2NywxMy45ODUsNi4wOTUKczA1NyxtMDEyLGNvbnRyb2wgLDQuNjM5LDMuMDcsMTEuMTU0LDYuNDk5CnMwNTgsbTAxMixjb250cm9sICw0LjU0OSwxLjY5NCwxMS44ODUsNi4xNgpzMDU5LG0wMTIsQ09OVFJPTCw0LjI2MywzLjIzNywxMi45OTksNS4zMjMKczA2MCxtMDEyLGNvbnRyb2wsMy41MzEsMi4zMzUsMTMuNTY1LDYuMTQKczA2MSxtMDEzLENvbnRyb2wsNC40MDYsMy4xMTYsLDcuNjQ0CnMwNjIsbTAxMyxDb250cm9sLDQuNDc1LDIuNTc2LDEzLjY2NSw2Ljc5NgpzMDYzLG0wMTMsQ09OVFJPTCw1LjgyMywxLjcyNSwxNC43NTEsOS41NDEKczA2NCxtMDEzLGNvbnRyb2wgLDUuNTY2LDMuNzA3LDE1LjcxNiw1LjM0NwpzMDY1LG0wMTMsY29udHJvbCAsNS4wNDIsMS44MzYsMTYuMDI0LDYuMTI5CnMwNjYsbTAxNCxjb250cm9sLDUuMDQ4LDMuODEyLDEwLjM5OCw2LjEwMQpzMDY3LG0wMTQsQ29udHJvbCwzLjM1NSwyLjAwOCw5LjkyNCw2LjM0OApzMDY4LG0wMTQsQ29udHJvbCw0LjYzOCw0LjYyNSwxMy40MTUsbi5kLgpzMDY5LG0wMTQsY29udHJvbCw1LjY1NSwzLjIwOSwxMy4zODIsOS4wNzEKczA3MCxtMDE0LENPTlRST0wsMy43NzIsMC41MjgsMTEuNTU2LDcuMTE5CnMwNzEsbTAxNSxDT05UUk9MLDQuODUzLDMuNjQ1LDEyLjkwNSw4Ljk5NgpzMDcyLG0wMTUsY29udHJvbCwyLjkyLDEuNDM3LDE0LjYwNCw3LjkxMgpzMDczLG0wMTUsQ29udHJvbCw0LjgwMSwyLjYxMiwxMi4wOCw1LjU5CnMwNzQsbTAxNSxDT05UUk9MLDMuNTU2LDEuOTc1LDExLjkxMyw3Ljk1NgpzMDc1LG0wMTUsY29udHJvbCAsNS40NTgsMy41NTksMTUuMDU2LDUuODM1CnMwNzYsbTAxNixDT05UUk9MLDQuMTc0LDIuNjE2LDE1LjQwNiw2LjQ3NwpzMDc3LG0wMTYsY29udHJvbCw0LjI0LDMuMjA0LDEyLjcyMyw3Ljc5OQpzMDc4LG0wMTYsY29udHJvbCw1LjQ2MywyLjAyMywxNC4yNzIsOC40NzEKczA3OSxtMDE2LGNvbnRyb2wsNS4yMTgsMS42MzksMTEuNTUsNS4yNDMKczA4MCxtMDE2LENvbnRyb2wsNS40MzEsNS42NzcsMTQuMzAyLDguMDk2CnMwODEsbTAxNyxDb250cm9sLDUuOTE0LDIuNjQxLDExLjYxOSxuLmQuCnMwODIsbTAxNyxDT05UUk9MLDMuODc0LDAuODgxLCw3LjcwOQpzMDgzLG0wMTcsY29udHJvbCAsNC41MjYsMS4yNjgsMTEuNzEyLDYuMDQ1CnMwODQsbTAxNyxjb250cm9sICwzLjc1OSwyLjc4NSwsNi43MjkKczA4NSxtMDE3LENvbnRyb2wsMy43NDMsMi43MzUsMTMuODA1LDQuNTk0CnMwODYsbTAxOCxDT05UUk9MLDQuNjQsMi42NDQsMTAuOTc0LDkuODkzCnMwODcsbTAxOCxjb250cm9sLCwsMTEuODE2LDkuMjE5CnMwODgsbTAxOCxDb250cm9sLDQuOTk2LDAuNDg3LDkuNDA2LDcuMTg0CnMwODksbTAxOCxjb250cm9sLDQuOTQ0LDIuOTUyLDE0LjMwNiw4Ljg4NApzMDkwLG0wMTgsQ09OVFJPTCw0LjEyNiwzLjI0MSwxMC4yNDUsNS4zODUKczA5MSxtMDE5LENvbnRyb2wsNS4zNDYsMy40OTEsMTIuMzYzLDUuMjc0CnMwOTIsbTAxOSxjb250cm9sICw1LjU1NywzLjA1MSwxMy42MTIsNS44ODYKczA5MyxtMDE5LENvbnRyb2wsNS42NzcsMi4zMjEsLDguMzY1CnMwOTQsbTAxOSxjb250cm9sICw1LjYwNywsOC43OTgsNS4zMzgKczA5NSxtMDE5LGNvbnRyb2wgLDYuNjQ5LCwxMi4wNTIsNS4zNTYKczA5NixtMDIwLENPTlRST0wsMy43MDgsMy4xNSwxMy43MzEsNC4xMjMKczA5NyxtMDIwLENvbnRyb2wsNS41ODgsMi45NzcsMTcuNjcyLDcuODAzCnMwOTgsbTAyMCxDb250cm9sLDQuNjkzLDMuMjcxLDEwLjExMiw1LjM1MQpzMDk5LG0wMjAsY29udHJvbCw1LjgxNCwyLjQyNiwxMy4wNTEsNi40MzYKczEwMCxtMDIwLENPTlRST0wsNi4yOTIsMi44NzUsLDguMTQ5CnMxMDEsbTAyMSxjb250cm9sICw2LjQzNSwyLjYzMSwxNC4wODUsOC4zMjYKczEwMixtMDIxLENPTlRST0wsMi43NjcsMy4zMDIsOS4xMDMsNy4zOTIKczEwMyxtMDIxLENvbnRyb2wsNC42MTYsMi41MjMsMTEuNzY1LG4uZC4KczEwNCxtMDIxLGNvbnRyb2wsMy4yNDMsMS41ODQsMTMuMTEyLDkuNTg3CnMxMDUsbTAyMSxjb250cm9sLDUuNTk5LDMuNDM3LDEzLjIzNCw0Ljk1MQpzMTA2LG0wMjIsY29udHJvbCAsNC4zODcsMC41NDYsMTEuMjk1LDQuNzgzCnMxMDcsbTAyMixDb250cm9sLDMuNjg3LDIuOTU4LDEwLjU0Niw1LjI3OApzMTA4LG0wMjIsQ09OVFJPTCw0LjQ4Nyw1LjQwNSwxMi44MjYsNC4wODMKczEwOSxtMDIyLGNvbnRyb2wgLDcuMDA5LDIuNjA3LDE0LjQwMSw4LjY2NQpzMTEwLG0wMjIsY29udHJvbCw1LjYyNywzLjQ4OSwxMi40MDUsNi43MjQKczExMSxtMDIzLENPTlRST0wsNS4zNDMsMS42NDEsMTAuNzk5LDYuOTM5CnMxMTIsbTAyMyxDb250cm9sLDUuMDI0LDIuNjA3LDEyLjk3Myw2Ljg0NApzMTEzLG0wMjMsQ29udHJvbCwzLjY4LDIuMTM5LDExLjgxOSw3LjE3OQpzMTE0LG0wMjMsY29udHJvbCAsMi44NzgsMy44NDEsOS45NTUsNS42NDcKczExNSxtMDIzLENPTlRST0wsNi4wOCwyLjIwNSwxMC42NDksNS4xNjQKczExNixtMDI0LENPTlRST0wsNi4xNDksMi44MDUsMTMuOTAzLDguMzc4CnMxMTcsbTAyNCxDT05UUk9MLDUuOTY2LDEuNTksMTAuNTk0LDcuMzc3CnMxMTgsbTAyNCxjb250cm9sICw1LjIzNCw1LjI5NCwxMS43NDEsNy41NQpzMTE5LG0wMjQsQ09OVFJPTCw1LjA1NywyLjQ2MSwxNS44MTgsbi5kLgpzMTIwLG0wMjQsQ09OVFJPTCw1LjM2Myw0LjQ2LDEyLjQzMyw0Ljg4MwpzMTIxLG0wMjUsQ29udHJvbCw1LjAxLDMuNjU2LDEwLjkyOSw0LjkyMgpzMTIyLG0wMjUsY29udHJvbCAsNC42MjgsMy4yMTIsOS41NDcsNy44MDkKczEyMyxtMDI1LGNvbnRyb2wsNS40NjUsMi44MTksMTMuNDQ2LDcuMjA2CnMxMjQsbTAyNSxjb250cm9sLDUuNTQ1LDIuNDI4LDEwLjI4NSw4LjQ5NwpzMTI1LG0wMjUsY29udHJvbCw1LjAwNCwsMTEuMTE4LDUuNTg4CnMxMjYsbTAyNixjb250cm9sICw0LjQzMywxLjU1OSwxMS4zNDQsNi43ODQKczEyNyxtMDI2LENvbnRyb2wsNC4xNDksMi44NTcsNS4yNTgsNy4yNzYKczEyOCxtMDI2LENvbnRyb2wsNS4yMTEsMy41OTMsOS44NDksNC44NjYKczEyOSxtMDI2LGNvbnRyb2wsMy41NjksMy40ODUsOS42ODksNS40ODIKczEzMCxtMDI2LENvbnRyb2wsNS4yMzksNC4wNjcsMTAuNzI2LDYuNjMxCnMxMzEsbTAyNyxjb250cm9sICw0LjU2NiwyLjIyMiwxMy40MTcsNC42NDkKczEzMixtMDI3LENPTlRST0wsMy45ODksMi40OTgsMTAuMjA5LDMuMzQ1CnMxMzMsbTAyNyxjb250cm9sICw0LjQ2MywzLjUwNCwxNC44OCw2LjcxOApzMTM0LG0wMjcsQ09OVFJPTCwsMi43NDcsMTEuMTQsNi43OTMKczEzNSxtMDI3LGNvbnRyb2wsNS42MjIsMS42NDQsMTQuNjAyLDcuNzU5CnMxMzYsbTAyOCxDT05UUk9MLDIuODQyLCw5Ljg4OCw3Ljg2MgpzMTM3LG0wMjgsQ29udHJvbCw2LjI0MSwyLjkyNCwxMS42OTEsNy42NTYKczEzOCxtMDI4LENPTlRST0wsMi45OTMsMi42NTcsMTAuNzU0LDcuNzgxCnMxMzksbTAyOCxjb250cm9sICw1LjgzNSwyLjg5OCwxMy41NTEsMTAuMzI4CnMxNDAsbTAyOCxDT05UUk9MLDYuNjQ5LDQuMDM4LDE0LjEyOSw2LjQ1MQpzMTQxLG0wMjksY29udHJvbCAsNC45NjUsNC4yNDIsOS45MDEsNy43MDMKczE0MixtMDI5LGNvbnRyb2wsMy41MjIsMS41NTIsMTMuMDc1LG4uZC4KczE0MyxtMDI5LGNvbnRyb2wgLDYuNzQ0LDIuODM4LDExLjI3MSw2LjY1NgpzMTQ0LG0wMjksY29udHJvbCAsNS4xODQsMi4wNjEsLDguNjMKczE0NSxtMDI5LENPTlRST0wsMi4zOSwyLjExNCwxMi44OTYsOC42ODEKczE0NixtMDMwLENPTlRST0wsNi41MzYsMi4wNjksLDcuMzE1CnMxNDcsbTAzMCxDb250cm9sLDIuMjQ5LDQuMDAyLDExLjUwNiw5Ljg2MgpzMTQ4LG0wMzAsQ09OVFJPTCw1LjI5NywyLjEzNiw5LjQzNCw3LjYwOApzMTQ5LG0wMzAsY29udHJvbCAsNC40NzUsMi4wNDEsMTQuMDcyLDYuNjQyCnMxNTAsbTAzMCxDb250cm9sLDUuMDQxLDIuNTc2LDEzLjgzNyw4LjY3MwpzMTUxLG0wMzEsdHJlYXRtZW50ICw0LjA2OCwyLjYxNyw5Ljk3NiwxMC4yODIKczE1MixtMDMxLHRyZWF0bWVudCAsNi4zMzcsMi40MTMsMTEuMTQ1LG4uZC4KczE1MyxtMDMxLHRyZWF0bWVudCAsNy40MzYsMS43NjcsOS4zNTEsOS42MTYKczE1NCxtMDMxLFRyZWF0bWVudCw1LjQ2OCw0LjExMywxMy41OCw1LjQ2MwpzMTU1LG0wMzEsdHJlYXRtZW50LDYuMjc3LDIuNzQzLDExLjM3OSw3Ljk1NApzMTU2LG0wMzIsdHJlYXRtZW50ICw1LjkzMSwzLjgxOCwxNC40MDYsNi4zMTUKczE1NyxtMDMyLFRSRUFUTUVOVCw1Ljg4OSwxLjQ2NiwxMy42MTYsOC4zNDcKczE1OCxtMDMyLHRyZWF0bWVudCw2LjUzOCwyLjcwNiwxMy40ODEsOC45NDUKczE1OSxtMDMyLFRyZWF0bWVudCw1Ljg0NiwzLjE2NiwxMC45NjIsNy4wMTYKczE2MCxtMDMyLFRyZWF0bWVudCw2Ljk5OCwyLjg3LDExLjQzMyw4LjQwNQpzMTYxLG0wMzMsdHJlYXRtZW50ICw1Ljc1MiwzLjkxMiwxMi43OTIsNy4xODYKczE2MixtMDMzLFRyZWF0bWVudCw3LjU5NiwzLjEwMSwsNi4zMDYKczE2MyxtMDMzLFRSRUFUTUVOVCw3LjU0NSwzLjcwNSwxMS45NzcsNC4wMjQKczE2NCxtMDMzLFRSRUFUTUVOVCw0LjkyMiwzLjA0OSw4LjE4LDguOTEzCnMxNjUsbTAzMyx0cmVhdG1lbnQgLDYuMjU3LDMuMTM2LDkuODQsNS45MzUKczE2NixtMDM0LFRyZWF0bWVudCw1LjY1NCwyLjgzNywxMS40MjUsNy4wNzYKczE2NyxtMDM0LHRyZWF0bWVudCAsNS44ODEsLTAuMDIxLDkuNjkxLDYuNTEyCnMxNjgsbTAzNCxUUkVBVE1FTlQsLDIuNjQ4LDExLjUxNSw3LjczMwpzMTY5LG0wMzQsVFJFQVRNRU5ULDcuMDAzLDMuMDc4LDE0LjM5NywxMC4wOTQKczE3MCxtMDM0LHRyZWF0bWVudCw2LjAwMSwxLjAzMiwxMy45MTEsNi4wMDgKczE3MSxtMDM1LHRyZWF0bWVudCAsNi4yNCwyLjQ1OSwxMC42MDMsNy40MDIKczE3MixtMDM1LFRSRUFUTUVOVCw4LjA0LDIuMzIzLDEzLjc0NCw0LjM5NgpzMTczLG0wMzUsVFJFQVRNRU5ULDYuNjk3LDMuNzU4LDExLjc2OCw4LjkwNApzMTc0LG0wMzUsVFJFQVRNRU5ULDcuMTMsNS4yNDYsOC40MjcsNi4xNDIKczE3NSxtMDM1LFRyZWF0bWVudCw3LjQ3NiwxLjg3MSwxNi44NDIsNy42MzgKczE3NixtMDM2LHRyZWF0bWVudCw1LjYsMy41NjMsMTIuNTA4LDYuNDQKczE3NyxtMDM2LHRyZWF0bWVudCAsOC4xMjQsNC42MzMsMTIuNDE4LDEwLjQ1NwpzMTc4LG0wMzYsdHJlYXRtZW50LDQuNjIxLDMuNjU4LDEwLjQ5Nyw5LjYzNgpzMTc5LG0wMzYsdHJlYXRtZW50LDcuNzM2LDIuNzcsMTIuMzgzLDkuMTIKczE4MCxtMDM2LFRSRUFUTUVOVCw3Ljg2NCwyLjQ0LDkuODk0LDcuODY4CnMxODEsbTAzNyxUUkVBVE1FTlQsNy45OTQsMS40NjYsMTAuNjM3LDguMjQ4CnMxODIsbTAzNyxUUkVBVE1FTlQsNi44ODgsMS41MjcsMTMuMzgyLDguMDE4CnMxODMsbTAzNyx0cmVhdG1lbnQgLDcuODE2LDEuNjg5LDEwLjE1Nyw4LjgyNgpzMTg0LG0wMzcsdHJlYXRtZW50LDYuOTk0LDMuMTk0LDEwLjg3Miw3LjE1CnMxODUsbTAzNyxUUkVBVE1FTlQsNi4zMTQsNC4wOTYsNy4wMDYsOS4zNDQKczE4NixtMDM4LFRSRUFUTUVOVCw3LjQzMiwzLjY0NiwsOC41NQpzMTg3LG0wMzgsdHJlYXRtZW50ICw2LjExNCwxLjYwNCwxMi43MDUsNy4wNjQKczE4OCxtMDM4LHRyZWF0bWVudCw2LjY5NiwzLjczLDEwLjgwNCw3Ljc4MQpzMTg5LG0wMzgsdHJlYXRtZW50ICw2LjgxNywzLjQ1MSw4LjgwMyw1LjY4MgpzMTkwLG0wMzgsdHJlYXRtZW50ICw2LjM0LDMuMTcsMTIuNzM4LDYuODkzCnMxOTEsbTAzOSx0cmVhdG1lbnQgLDMuNTk1LDIuODYxLDEwLjkzNiw3LjI2CnMxOTIsbTAzOSxUUkVBVE1FTlQsNS40MTksMy42NCwxMi41MjgsOC41NjIKczE5MyxtMDM5LHRyZWF0bWVudCwsMy44LDEzLjA3Nyw1LjkxMwpzMTk0LG0wMzksdHJlYXRtZW50ICwsMy4wNzksMTUuMTYsNi45NzcKczE5NSxtMDM5LFRyZWF0bWVudCwsMC44NywxMC41NDksOC4zOApzMTk2LG0wNDAsdHJlYXRtZW50LDYuMjQsMy45NDYsMTEuMzY0LDkuNjEyCnMxOTcsbTA0MCxUcmVhdG1lbnQsNS43MDcsMi41MywxMi4yNjIsOC4xNjcKczE5OCxtMDQwLFRyZWF0bWVudCw0LjIwNiwyLjQxOCw4LjU1Myw2Ljk4MgpzMTk5LG0wNDAsVFJFQVRNRU5ULDMuOTQ2LDEuMywsMTAuNzQxCnMyMDAsbTA0MCxUcmVhdG1lbnQsLCwxMC4xMTMsNi45MzcKczIwMSxtMDQxLFRyZWF0bWVudCw2LjkyMSwzLjc4NCwxMi44MTEsNC4yNTgKczIwMixtMDQxLFRyZWF0bWVudCw2Ljg4MiwzLjIxLDEyLjQ1NSw4Ljc1NQpzMjAzLG0wNDEsVHJlYXRtZW50LDcuMTg2LDIuMTI0LDEyLjU1Niw3LjMzNgpzMjA0LG0wNDEsVHJlYXRtZW50LDYuNDA5LCwxMC42ODMsNi41CnMyMDUsbTA0MSx0cmVhdG1lbnQgLDYuMiw0LjM2Niw4LjMyNCw3Ljg3NwpzMjA2LG0wNDIsdHJlYXRtZW50ICw0Ljc1LDIuNjc4LDEwLjkwMyw1LjY5NwpzMjA3LG0wNDIsdHJlYXRtZW50ICw2LjQyOSwzLjczMiwxNS4wMDQsOC41MjMKczIwOCxtMDQyLHRyZWF0bWVudCw1LjgwMSwzLjIyNiw5LjY2Nyw2LjA0NgpzMjA5LG0wNDIsVFJFQVRNRU5ULDYuMjk2LDMuMTE1LDEyLjQyNCw2LjI2MwpzMjEwLG0wNDIsdHJlYXRtZW50LDQuMzYsMy4xNzgsMTQuMTQ2LDUuODE5CnMyMTEsbTA0Myx0cmVhdG1lbnQsNi44NTMsMi40MzgsOS4yMzYsOC45ODMKczIxMixtMDQzLHRyZWF0bWVudCw2LjU3NSwyLjA4NCw5LjYyMSw5LjE2NQpzMjEzLG0wNDMsVFJFQVRNRU5ULDUuNjg1LDMuMjI3LDExLjI2OSw4Ljg5NApzMjE0LG0wNDMsVHJlYXRtZW50LDQuMjY3LDIuNzk2LDExLjU0Miw4LjAKczIxNSxtMDQzLFRyZWF0bWVudCw0LjgxMSw0LjIyMywxMS44NDgsOC4zNDEKczIxNixtMDQ0LFRyZWF0bWVudCw2LjIwOCw0LjE1NywxMS41NSw3Ljc5MwpzMjE3LG0wNDQsVHJlYXRtZW50LDQuMzk0LDIuNjU0LDEyLjk2Niw2Ljc2NwpzMjE4LG0wNDQsVFJFQVRNRU5ULDUuOTg1LDIuNzk0LDguNDIsOC43MDYKczIxOSxtMDQ0LFRyZWF0bWVudCw2LjQ1LDIuNjkzLDEwLjg5OCw3LjYKczIyMCxtMDQ0LFRSRUFUTUVOVCw2Ljc4MSwzLjY0LDEwLjcxOSw3LjI1MwpzMjIxLG0wNDUsdHJlYXRtZW50ICw2Ljg5MywxLjY0NSwxMC4wMjksOC4wNjkKczIyMixtMDQ1LHRyZWF0bWVudCw0LjU5NCwzLjIyMiw5LjgxMyw1Ljc5NQpzMjIzLG0wNDUsVFJFQVRNRU5ULDUuMDkxLCwsOC41MTEKczIyNCxtMDQ1LHRyZWF0bWVudCAsNi40MDcsMi4xMjcsMTEuMDE4LDcuNjM5CnMyMjUsbTA0NSxUcmVhdG1lbnQsNi4wNjcsMS41MiwxNC45NzksNS40NDQKczIyNixtMDQ2LHRyZWF0bWVudCAsNi45MzEsLDEzLjY3NCw1LjMxNwpzMjI3LG0wNDYsdHJlYXRtZW50LDMuNzQ2LDMuOTMsOS4xNzIsNi4zODQKczIyOCxtMDQ2LFRSRUFUTUVOVCw2Ljg1NywzLjIzMywxMy42NzQsOC4zODQKczIyOSxtMDQ2LFRSRUFUTUVOVCw1LjYwNCwyLjc5OCwxMy4wMTUsOC41MzYKczIzMCxtMDQ2LHRyZWF0bWVudCw3LjAyOCwyLjExNyw5LjM0Myw1LjUyNApzMjMxLG0wNDcsVFJFQVRNRU5ULCwyLjU5MiwxMy4zNTcsOC44ODkKczIzMixtMDQ3LFRSRUFUTUVOVCw1LjM4OSwzLjM0NCw5LjU0MSw2LjQxNQpzMjMzLG0wNDcsdHJlYXRtZW50ICw1LjMzLDEuOTk3LDEyLjYyOSw1LjkxOApzMjM0LG0wNDcsdHJlYXRtZW50LDYuMDg2LDIuNjY4LDEzLjEwOSw4LjUzNQpzMjM1LG0wNDcsVHJlYXRtZW50LDUuNDc3LDMuNzgyLDEyLjQ4Miw2LjA4MQpzMjM2LG0wNDgsVHJlYXRtZW50LDUuNTUzLDAuNDQxLDkuNjg5LDYuODMyCnMyMzcsbTA0OCxUcmVhdG1lbnQsNC42MjcsMi42MTUsMTIuNDYsNS42OQpzMjM4LG0wNDgsVHJlYXRtZW50LDYuNDIyLDAuOTM4LDEyLjYxNiw2LjU1CnMyMzksbTA0OCx0cmVhdG1lbnQgLDcuMjczLDMuNDI5LDEzLjUwMSw3Ljg2MQpzMjQwLG0wNDgsdHJlYXRtZW50ICw2LjUwOSwyLjU3OSwxMi40NDQsNy4yNjkKczI0MSxtMDQ5LHRyZWF0bWVudCAsNC42NDMsMi42OSwxMy4xMTUsNy41NzkKczI0MixtMDQ5LFRyZWF0bWVudCw2LjQ4MSwyLjQwNiwxMi4zMDEsNy40MzQKczI0MyxtMDQ5LHRyZWF0bWVudCAsNC42Miw0LjcyNSwxMC43ODYsNi44OTIKczI0NCxtMDQ5LHRyZWF0bWVudCw2LjE5MiwzLjI5MywxMi4xODUsNy43MjUKczI0NSxtMDQ5LHRyZWF0bWVudCAsOC41LDMuMTI2LDEzLjYyNyw3LjI1MQpzMjQ2LG0wNTAsVHJlYXRtZW50LDYuNDk1LDMuOTEyLDkuNTM3LDMuNDQ3CnMyNDcsbTA1MCx0cmVhdG1lbnQsNy42NjcsNC4zMTgsOS43NjgsNC45ODQKczI0OCxtMDUwLFRyZWF0bWVudCw2LjM3NSwzLjUyNywxMy4wNDMsNy40MjMKczI0OSxtMDUwLFRSRUFUTUVOVCw2Ljk1NiwzLjcxOCwxMy45MTgsNi40MTcKczI1MCxtMDUwLFRyZWF0bWVudCw2Ljk5MywyLjg2MywxMC4yODIsOS42NTMKczI1MSxtMDUxLHRyZWF0bWVudCw3Ljc0MSwzLjYxOCwxMy40OTksNi4xNTcKczI1MixtMDUxLFRyZWF0bWVudCw2LjMwNCwzLjYyOCw4LjgzNiw2LjQyCnMyNTMsbTA1MSx0cmVhdG1lbnQsNy4xMjEsMi43OTQsMTEuMzY1LDguMTk3CnMyNTQsbTA1MSx0cmVhdG1lbnQsNi4yOTcsLDExLjEwNiw4LjA1MQpzMjU1LG0wNTEsVHJlYXRtZW50LDUuMzY5LDMuMzQzLDguNDcsOS41NjYKczI1NixtMDUyLFRyZWF0bWVudCw1Ljc1NSwxLjk4LDEwLjgyNCw4LjQ3OApzMjU3LG0wNTIsVFJFQVRNRU5ULDQuNjQ3LDIuODQ3LCw2LjMxOApzMjU4LG0wNTIsVFJFQVRNRU5ULDQuMTUzLDMuMTY5LDEzLjMyMiw4LjE4MwpzMjU5LG0wNTIsVFJFQVRNRU5ULDYuNzgxLDUuNzc2LDEyLjI2LDguMTk2CnMyNjAsbTA1Mix0cmVhdG1lbnQgLDQuNzc3LDEuNTg2LDE1LjQsMTAuNzY4CnMyNjEsbTA1Myx0cmVhdG1lbnQsNS4xMywyLjIzNiwxMi4yNjUsNS41NDQKczI2MixtMDUzLFRSRUFUTUVOVCw1LjMzMywzLjc3LDExLjgxNyw2LjIyMgpzMjYzLG0wNTMsdHJlYXRtZW50ICw3LjA4NywxLjc4NywxMS44NTEsOC44NjMKczI2NCxtMDUzLFRSRUFUTUVOVCw2LjU1OSwzLjExMiwxMi43MjksNS45NjMKczI2NSxtMDUzLHRyZWF0bWVudCw1LjU2MiwtMC4wMjcsMTEuMzA2LDcuMjI2CnMyNjYsbTA1NCx0cmVhdG1lbnQgLDUuOTQ1LDMuMDcsMTIuMjcsOS44MTMKczI2NyxtMDU0LFRyZWF0bWVudCw2LjkxOCwyLjI4LDEyLjM5MSw3LjgxNwpzMjY4LG0wNTQsVHJlYXRtZW50LDUuODQ4LDEuMTc1LDkuMDc1LDguMDYxCnMyNjksbTA1NCx0cmVhdG1lbnQsNS44MzQsMi4yMDQsMTEuOTQsOC4yNTMKczI3MCxtMDU0LFRyZWF0bWVudCw3LjkzNiwyLjg3MywxMS40MjgsNy44MTcKczI3MSxtMDU1LFRyZWF0bWVudCw3LjAzNSwxLjg0NSwxMi40ODksNS43MjgKczI3MixtMDU1LFRSRUFUTUVOVCw1LjUyMiw1LjA4OCwxMS43NTYsNi4zNjUKczI3MyxtMDU1LHRyZWF0bWVudCAsNi41NjgsMi4xNjIsMTIuNTg1LDUuMzU4CnMyNzQsbTA1NSx0cmVhdG1lbnQgLDUuNjM5LDMuNDg0LDcuMzksOC41MjQKczI3NSxtMDU1LFRyZWF0bWVudCw3LjM0MSwyLjkxNCwxMy4zMyw1Ljk4NApzMjc2LG0wNTYsdHJlYXRtZW50LDUuOTk2LDIuNzIxLDExLjU5OCw5LjQ5MQpzMjc3LG0wNTYsVFJFQVRNRU5ULDYuMzQ1LDMuMDQsMTEuOTM2LDguMDg4CnMyNzgsbTA1NixUUkVBVE1FTlQsNC4xNjcsMi4xNDQsMTMuODI1LDcuMDMyCnMyNzksbTA1Nix0cmVhdG1lbnQgLDYuMzY5LC0wLjM0Miw5LjAwNSxuLmQuCnMyODAsbTA1NixUUkVBVE1FTlQsNS4yMTcsMi4yNDksMTAuNzU4LDYuMjUKczI4MSxtMDU3LHRyZWF0bWVudCwsNC4xNjEsMTQuMDA0LDcuMzQKczI4MixtMDU3LHRyZWF0bWVudCw1LjA2LDUuMjA1LDEwLjM2Nyw5LjYwNwpzMjgzLG0wNTcsdHJlYXRtZW50ICwsMS45NTIsMTEuNDA1LDYuNTIyCnMyODQsbTA1Nyx0cmVhdG1lbnQsNC41NTMsMy4xOTEsMTAuNDA4LDguNjAyCnMyODUsbTA1Nyx0cmVhdG1lbnQsNi4xMDgsMi44ODIsMTEuNDY3LDcuMzEKczI4NixtMDU4LHRyZWF0bWVudCAsOC41MjQsMS43MjgsOS40NzQsNi44NzIKczI4NyxtMDU4LFRyZWF0bWVudCw0Ljg3NSwxLjQ1MywxMC43MzIsNi42MjQKczI4OCxtMDU4LHRyZWF0bWVudCw2LjM2MSwxLjU2Myw5LjA1LDguNzg1CnMyODksbTA1OCx0cmVhdG1lbnQsNC42MjcsMy44MTEsOS43MjMsNS43NDYKczI5MCxtMDU4LFRSRUFUTUVOVCw2LjkwOCwzLjg3OCw1LjgyNSw4LjY5MgpzMjkxLG0wNTksVHJlYXRtZW50LDUuNTc2LDMuODE1LDEzLjI4LDcuODYzCnMyOTIsbTA1OSx0cmVhdG1lbnQsNi43MDksLDEwLjY4Niw1LjgxNwpzMjkzLG0wNTksdHJlYXRtZW50ICw2LjYyMSwyLjU2MiwxMC4zNCwxMC41NTQKczI5NCxtMDU5LHRyZWF0bWVudCAsNS42NjUsMS44MzEsNS4zNzEsOC41NzEKczI5NSxtMDU5LHRyZWF0bWVudCAsLDMuMzEsMTAuMDY2LDcuNDUzCnMyOTYsbTA2MCx0cmVhdG1lbnQgLDUuODksMy41NjcsMTEuMTAxLDYuMzUKczI5NyxtMDYwLFRSRUFUTUVOVCw3LjAzLDEuODc1LDEzLjM2OCw1LjUwNwpzMjk4LG0wNjAsdHJlYXRtZW50ICwzLjc4MSwyLjE0OCwxMi41OTYsOC45OTMKczI5OSxtMDYwLHRyZWF0bWVudCAsNi4yMzEsMC4yMjMsMTMuMDksNS4xNjEKczMwMCxtMDYwLHRyZWF0bWVudCw1LjIxNSw0LjQ5NCwxNC4xMzMsNS43NjMK',
    'a12pm_mouse_metadata.csv':
        'bW91c2VfaWQsc2V4LGFnZV93ZWVrcyxiYXRjaAptMDAxLE0sMjksYmF0Y2gxCm0wMDIsRiwxMixiYXRjaDEKbTAwMyxGLDI4LGJhdGNoMQptMDA0LE0sMjcsYmF0Y2gxCm0wMDUsRiwyOSxiYXRjaDEKbTAwNixGLDI1LGJhdGNoMQptMDA3LEYsMTksYmF0Y2gxCm0wMDgsRiwyMSxiYXRjaDEKbTAwOSxNLDEyLGJhdGNoMQptMDEwLEYsMTUsYmF0Y2gxCm0wMTEsTSwxOCxiYXRjaDEKbTAxMixNLDI4LGJhdGNoMQptMDEzLE0sMjUsYmF0Y2gxCm0wMTQsTSwyMCxiYXRjaDEKbTAxNSxNLDIxLGJhdGNoMQptMDE2LEYsMTcsYmF0Y2gxCm0wMTcsTSwxNyxiYXRjaDEKbTAxOCxNLDI3LGJhdGNoMQptMDE5LEYsOCxiYXRjaDEKbTAyMCxGLDE1LGJhdGNoMQptMDIxLEYsMTYsYmF0Y2gyCm0wMjIsTSwyMyxiYXRjaDIKbTAyMyxNLDE3LGJhdGNoMgptMDI0LE0sMTQsYmF0Y2gyCm0wMjUsRiw5LGJhdGNoMgptMDI2LE0sMTMsYmF0Y2gyCm0wMjcsTSwxMCxiYXRjaDIKbTAyOCxNLDIzLGJhdGNoMgptMDI5LEYsMTcsYmF0Y2gyCm0wMzAsTSwxMyxiYXRjaDIKbTAzMSxNLDE5LGJhdGNoMgptMDMyLE0sMTgsYmF0Y2gyCm0wMzMsTSwxOCxiYXRjaDIKbTAzNCxGLDIwLGJhdGNoMgptMDM1LEYsMjEsYmF0Y2gyCm0wMzYsRiwxMixiYXRjaDIKbTAzNyxGLDExLGJhdGNoMgptMDM4LEYsMjQsYmF0Y2gyCm0wMzksRiwyOSxiYXRjaDIKbTA0MCxGLDIwLGJhdGNoMgptMDQxLE0sMTcsYmF0Y2gzCm0wNDIsTSwyMSxiYXRjaDMKbTA0MyxGLDgsYmF0Y2gzCm0wNDQsRiwxNixiYXRjaDMKbTA0NSxGLDE1LGJhdGNoMwptMDQ2LE0sMTcsYmF0Y2gzCm0wNDcsTSwyNCxiYXRjaDMKbTA0OCxGLDE4LGJhdGNoMwptMDQ5LEYsMjYsYmF0Y2gzCm0wNTAsRiwxOCxiYXRjaDMKbTA1MSxNLDgsYmF0Y2gzCm0wNTIsRiwyMixiYXRjaDMKbTA1MyxGLDI1LGJhdGNoMwptMDU0LEYsMjAsYmF0Y2gzCm0wNTUsRiwyOCxiYXRjaDMKbTA1NixNLDE3LGJhdGNoMwptMDU3LEYsMjUsYmF0Y2gzCm0wNTgsRiw4LGJhdGNoMwptMDU5LE0sOCxiYXRjaDMKbTA2MCxGLDI1LGJhdGNoMwo=',
}

for _name, _b64 in _DATA.items():
    pathlib.Path(_name).write_bytes(base64.b64decode(_b64))

print('Created:', ', '.join(_DATA))

In [ ]:
import importlib.util, os

packages = ['numpy', 'pandas', 'matplotlib', 'scipy', 'sklearn']
files = ['a12pm_measurements.csv', 'a12pm_mouse_metadata.csv']  # a12pm_hw2.py is defined in a cell below, not a file

for pkg in packages:
    print(f'{pkg:30s}', 'OK' if importlib.util.find_spec(pkg) is not None else 'MISSING')
for f in files:
    print(f'{f:30s}', 'OK' if os.path.exists(f) else 'MISSING')


1. [Loading data with pandas](#loading)
1. [Inspecting what you loaded](#inspecting)
1. [Cleaning](#cleaning)
1. [Selecting and filtering](#selecting)
1. [Missing data](#missing)
1. [Group summaries with groupby](#groupby)
1. [Joining tables with merge](#merge)
1. [DataFrame → numpy](#bridge)
1. [scipy.stats: statistics in python](#scipy)
1. [scikit-learn in ten minutes](#sklearn)
1. [Where to go for your own field](#signposts)
1. [Homework](#homework)

<a id="loading"></a>

## 1. Loading data with pandas

<a href="https://pandas.pydata.org/">pandas</a> is the standard Python tool for tabular data — data that lives in rows and columns. If you are familiar with R, it's just like that. Pandas comes with Anaconda, so you already have it if you followed the instructions from Lecture 1.  The standard nickname for pandas is `pd`:

In [ ]:
import numpy as np
import pandas as pd

This repository directory contains a small (fake, but realistic) lab dataset: gene expression measurements from 60 mice, 5 tissue samples per mouse, half the mice given a treatment.  It has the kinds of problems representative of real datasets.  Load it with `read_csv`:

In [ ]:
df = pd.read_csv('a12pm_measurements.csv')
df.head()

`df` is a **DataFrame** — pandas' central object.  Think of it as a spreadsheet that Python can talk to: unlike a numpy array, its columns have **names**, and different columns can hold different types (text, numbers, dates).

`head()` shows the first five rows.  `shape` tells you how big it is:

In [ ]:
df.shape

<a id="inspecting"></a>

## 2. Inspecting what you loaded

**Never trust a freshly loaded file.**  Three commands tell you most of what you need to know.  First, `dtypes` — what type is each column?

In [ ]:
df.dtypes

Look at `gene_d`.  The other gene columns are `float64` (numbers), but `gene_d` is not — pandas is storing it as **text**.  Why?  Because a few entries in that column say `n.d.` ("not detected" — a lab-notebook habit), and one text entry is enough to turn the whole column into text.  We will fix it in a moment.

Next, `describe` — summary statistics for every numeric column:

In [ ]:
df.describe()

(Notice `gene_d` doesn't appear — pandas doesn't compute statistics on text.)

Third: how much is missing?  `isna()` marks every missing cell, and `.sum()` counts them per column:

In [ ]:
df.isna().sum()

Finally, for any categorical column, always look at `value_counts()`:

In [ ]:
df['condition'].value_counts()

There are only two experimental conditions, but the computer sees eight different labels — `control`, `Control`, `CONTROL`, and a `control ` with a trailing space you can't even see.  A human reads these as the same word.  **Software does not.**  This single problem — same category, different strings — has ruined more analyses than any other item in this notebook.

<a id="cleaning"></a>

## 3. Cleaning

We can easily fix the issues with the dataset. First, normalize the condition labels: lowercase them and strip surrounding whitespace.  The `.str` accessor applies a string operation to the whole column at once:

In [ ]:
df['condition'] = df['condition'].str.lower().str.strip()
df['condition'].value_counts()

Two clean groups of 150.

Second, `gene_d`: convert it to numbers with `pd.to_numeric`.  The option `errors='coerce'` says: anything that can't be parsed as a number (like `n.d.`) becomes NaN — which is the truth, since those values were not detected:

In [ ]:
df['gene_d'] = pd.to_numeric(df['gene_d'], errors='coerce')
df.dtypes

<a id="selecting"></a>

## 4. Selecting and filtering

One column (this gives you a **Series** — a single labeled column):

In [ ]:
df['gene_a']

Several columns — note the double brackets (you are passing a *list* of names):

In [ ]:
df[['gene_a', 'gene_b']].head()

Rows are selected two ways.  `iloc` selects by **position** (like numpy indexing), `loc` by **label and condition**:

In [ ]:
print(df.iloc[0])       # first row
df.loc[df['gene_a'] > 7].head()   # all rows where gene_a > 7

That second pattern — a True/False condition inside `df.loc[...]` — is the pandas equivalent of "SELECT ... WHERE" (if you are familiar with database terminology), and you will use it constantly:

In [ ]:
treated = df.loc[df['condition'] == 'treatment']
control = df.loc[df['condition'] == 'control']
print(len(treated), 'treated samples;', len(control), 'control samples')

<a id="missing"></a>

## 5. Missing data

Datasets often have missing data. What you do about it is a research design question: your choice depends on *why* the data is missing:

1. **Drop the rows** (`dropna`).  Defensible when little is missing and it's missing at random.  Costs you samples.
2. **Fill with a neutral value** (`fillna`), e.g. the column median.  Defensible when you must keep every row.  Invents data — say so in your methods section.
3. **Leave it and use methods that tolerate NaN** (e.g. `np.nanmean`, and most of `scipy.stats` has `nan_policy` options).

What is **never** defensible is not knowing which one you did, not having a reason for your choice, and not reporting what choice you made.

In [ ]:
print('rows before dropna:', len(df))
print('rows after dropna: ', len(df.dropna()))
print('gene_a median-filled mean:', df['gene_a'].fillna(df['gene_a'].median()).mean().round(3))

<a id="groupby"></a>

## 6. Group summaries with groupby

We usually want to summarize our data, such as getting the "Mean expression per condition". Once you have your df set up, doing this in pandas is one line. `groupby` splits the table by a categorical column, applies a computation to each group, and reassembles the result:

In [ ]:
df.groupby('condition')['gene_a'].mean()

In [ ]:
gene_cols = ['gene_a', 'gene_b', 'gene_c', 'gene_d']
df.groupby('condition')[gene_cols].mean()

Reading across that table: `gene_a` looks strongly affected by treatment, `gene_b` looks untouched.  In Section 9 we'll ask whether the difference is statistically credible.

Also, pandas connects straight back to the matplotlib we used earlier.  Every DataFrame and Series has a `.plot` accessor. You can use `.plot.bar()`, `.plot.hist()`, `.plot.scatter()` to draw the table it's called on, labels and all.  So a plot of the table we just made is one line:

In [ ]:
df.groupby('condition')[gene_cols].mean().plot.bar(title='Mean expression by condition', rot=0);
df.groupby('condition')[gene_cols].mean().T.plot.bar(title='Mean expression by gene', rot=0);

<a id="merge"></a>

## 7. Joining tables with merge

Real data often lives in **more than one data file or table**.  Here, we have per-mouse information (sex, age, batch) that is in a separate file, because it was recorded once per mouse, not once per sample:

In [ ]:
metadata = pd.read_csv('a12pm_mouse_metadata.csv')
metadata.head()

`merge` joins two tables on a shared key column — here `mouse_id`.  Every sample row picks up its mouse's metadata.  We give the merged table a **new name**, `df_full`, so this cell is safe to re-run — merging a table into itself a second time would create duplicate columns:

In [ ]:
df_full = df.merge(metadata, on='mouse_id')
df_full.head()

We won't go deeper today, but `merge` is worth additional study. It is how measurement files combine with sample sheets. It has options (`how='left'`, `'inner'`, `'outer'`) that control what happens to rows that don't match.  See the <a href="https://pandas.pydata.org/docs/user_guide/merging.html">pandas merging guide</a>.

<a id="bridge"></a>

## 8. DataFrame → numpy

**This is the most important cell in the notebook.**  Models don't directly take pandas DataFrames as input. They take numeric arrays (e.g. numpy arrays) as input.

1. choose your feature columns,
2. drop rows with missing features (a deliberate, visible choice — option 1 from Section 5),
3. `.to_numpy()` for the features → $X$,
4. pull out the label column → $y$.

In [ ]:
feature_cols = ['gene_a', 'gene_b', 'gene_c', 'gene_d']

clean = df_full.dropna(subset=feature_cols)
X = clean[feature_cols].to_numpy()
y = (clean['condition'] == 'treatment').to_numpy().astype(int)

print('X:', X.shape, X.dtype, '   y:', y.shape, y.dtype)
print(X[0:5,:])  # print the first five rows in X
print(X[-5:,:])  # print the last five rows in X
print('samples per class:', np.bincount(y))
print(y[0:5])  # print the first five rows in y
print(y[-5:])  # print the last five rows in y

250 samples, 4 features, and a 0/1 label for control/treatment.  From here on, everything in this workshop — today's t-test, tomorrow's neural network, Friday's protein language model — is some function of an $(X, y)$ pair shaped exactly like this.

<a id="scipy"></a>

## 9. scipy.stats: statistics in python

<a href="https://scipy.org/">scipy</a> is numpy's big sibling: a general toolbox of scientific computing routines, organized into submodules. These include scipy.stats (statistical tests and distributions), scipy.optimize (curve fitting), scipy.signal (filtering and spectral analysis), scipy.spatial (distances between data points), and scipy.ndimage (image processing), among others. Today we only need one of them: scipy.stats contains some common statistical functions: t-tests, correlations, ANOVA, and hundreds more. A lot of the same stuff as R; only the syntax changes.

We will use scipy.stats to do some simple statistics on our dataset. Is the treatment effect on `gene_a` credible?

In [ ]:
from scipy import stats

a = control['gene_a'].dropna()
b = treated['gene_a'].dropna()
result = stats.ttest_ind(a, b)
print('gene_a: t =', result.statistic.round(2), '  p =', result.pvalue)

We can test all four genes and then correct for multiple comparisons. `false_discovery_control` is the Benjamini–Hochberg FDR correction commonly used in genomics:

In [ ]:
pvals = []
for g in gene_cols:
    res = stats.ttest_ind(control[g].dropna(), treated[g].dropna())
    pvals.append(res.pvalue)

qvals = stats.false_discovery_control(pvals)

for g, p, q in zip(gene_cols, pvals, qvals):
    print(f'{g}:  p = {p:.2e}   FDR-adjusted p = {q:.2e}')

Notice the correction barely affects the very strong (A) and very weak results (B). It's the borderline p-values (C and D) that are affected most.

Correlations work the same way — `stats.pearsonr` and `stats.spearmanr`.  For example, `gene_c` is positively associated with age:

In [ ]:
ok = df_full.dropna(subset=['gene_c'])
r, p = stats.pearsonr(ok['gene_c'], ok['age_weeks'])
print('gene_c vs age:  r =', r.round(3), '  p =', f'{p:.1e}')

One more scipy module to remember for later: `scipy.spatial.distance` (`pdist`, `cosine`) computes distances between rows of a matrix.  **We will want it tomorrow afternoon**, when we ask which samples — and which words, and which proteins — are *similar* to each other.

<a id="sklearn"></a>

## 10. scikit-learn in ten minutes

<a href="https://scikit-learn.org/">scikit-learn</a> is the standard Python library for classical machine learning.  Today we will only give a very short summary. But scikit-learn is nice, because it contains a huge number of different kinds of machine learning models. And they all rely on the same data structure shape (the one we've been using), and follow the same three-step pattern:

```
model.fit(X_train, y_train)      # learn from data
model.predict(X_test)            # apply to new data
model.score(X_test, y_test)      # how well did it do?
```

Remember that pattern. **The meaning of these steps is a focus of tomorrow morning's sessions.**

Two preliminary steps before any model uses the data.  First, split the samples into a **training set** (the model learns from these) and a held-out **test set** (we grade the model on these).  Second, **normalize** the features.  `StandardScaler` z-scores each feature column (subtract the column's mean and divide by its standard deviation), so every feature ends up with mean 0 and standard deviation 1.  We do this because models like logistic regression are sensitive to the scale of the features.  Without it, a gene measured in the thousands would dominate a model over one measured in fractions, even if the effect of the latter was greater.

For StandardScaler, `fit` just means "memorize each column's mean and std". Then `transform` applies the shift-and-divide.  Notice that we compute those means and stds from the *training* rows only — why that matters is for Thursday.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

scaler = StandardScaler().fit(X_train)   # fit on the training part only -- why? Thursday.
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print('training samples:', X_train.shape[0], '   test samples:', X_test.shape[0])

Now the model — the three-step pattern in action:

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_s, y_train)
print('accuracy on held-out samples:', round(model.score(X_test_s, y_test), 3))

A classifier that tells treated from control samples about 84% of the time, in five lines.  Two questions we might have:

* Why did we split the data before fitting?
* Is 84% *good*?

Both are tomorrow's 11am session.

Next, we can see how the fit-predict-score pattern is the same across many different kinds of models. Four models, four very different modeling philosophies, same three steps in python:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

models = [
    LogisticRegression(),                      # a linear decision rule
    KNeighborsClassifier(),                    # vote of the 5 most similar samples
    RandomForestClassifier(random_state=0),    # ensemble of decision trees
    LinearDiscriminantAnalysis(),              # the LDA of classical multivariate statistics
]

for model in models:
    model.fit(X_train_s, y_train)
    print(f'{type(model).__name__:30s} accuracy: {model.score(X_test_s, y_test):.3f}')

What you should **not** conclude from this table is that the highest number is "the best model" — with only 63 test samples, these differences are probably noise.  How to compare models honestly is something we will discuss tomorrow.

<a id="signposts"></a>

## 11. Where to go for your own field

| If you work with... | Look at |
|---|---|
| classical statistics, mixed models, GLMs | <a href="https://www.statsmodels.org/">statsmodels</a> (R-like formulas: `y ~ condition + age`) |
| sequences, alignments, genomics files | <a href="https://biopython.org/">biopython</a> |
| single-cell RNA-seq | <a href="https://scanpy.readthedocs.io/">scanpy</a> |
| EEG / MEG | <a href="https://mne.tools/">MNE</a> |
| MRI / neuroimaging | <a href="https://nipy.org/nibabel/">nibabel</a> |

Three habits that make any of these learnable:

1. **Read the Quick Start / tutorial page first**, not the API reference.  Every good library has one; it's written for exactly you.
2. **Install with one tool.**  Not all of them come pre-installed with Anaconda. But they are easy to install. You have two ways to do it: `conda install` and `pip install`.  Mixing them in one environment is the most common way scientific Python setups break.  Simple rule for this week: try `conda install` first; if the package isn't found, use `pip install`; when something is deeply broken, deleting and recreating the environment is cheaper than debugging it.
3. **Every library speaks numpy.**  Whatever field-specific loader you use, somewhere there is an array with your numbers in it — and from there, everything in this notebook applies.

<a id="homework"></a>

## Homework

The homework asks you to package what you just did into three reusable functions — the same three you will need for your own data.  As before, find the **✍️ Your turn** cell and fill in the functions there.

Start with `load_and_clean` and `to_feature_matrix` — we'll work on those here in the session.  `group_means` is yours to finish afterward.

### ✍️ Your turn: write `load_and_clean`, `to_feature_matrix`, `group_means`

**This is where you write code.**  Fill them in below, then run this cell (Shift+Enter).  The cells underneath use them, so come back and re-run this cell every time you change them.

In [ ]:
import numpy as np
import pandas as pd

def load_and_clean(path):
    '''
    Load a messy measurements CSV and clean it up.

    @param:
    path (str): path to a CSV file formatted like a12pm_measurements.csv

    @return:
    df (DataFrame): the loaded table, with two problems fixed:
      1. The `condition` column is lowercased and stripped of surrounding
         whitespace, so that it contains exactly two values:
         'control' and 'treatment'.
      2. The `gene_d` column is converted to numeric.  Entries that are not
         numbers (like 'n.d.') become NaN.
    Do not drop any rows: the returned DataFrame has the same number of rows
    as the file.

    Hints: .str.lower(), .str.strip(), pd.to_numeric(..., errors='coerce')
    '''
    raise NotImplementedError("You need to write this part!")

def to_feature_matrix(df, feature_cols, label_col):
    '''
    Convert a cleaned DataFrame into the (X, y) arrays that every model in
    this workshop eats.

    @param:
    df (DataFrame): a cleaned table, e.g. the output of load_and_clean
    feature_cols (list of str): names of the numeric feature columns
    label_col (str): name of the label column

    @return:
    X (ndarray of float, shape (n, len(feature_cols))): feature matrix
    y (ndarray, shape (n,)): label values (strings are fine)

    Rows that have NaN in ANY of the feature columns are dropped from both
    X and y (so the two arrays stay aligned).

    Hints: df.dropna(subset=...), .to_numpy()
    '''
    raise NotImplementedError("You need to write this part!")

def group_means(df, by, value):
    '''
    Compute the mean of one column, separately for each level of another.

    @param:
    df (DataFrame): the data table
    by (str): name of the categorical column to group by
    value (str): name of the numeric column to average

    @return:
    means (Series): the mean of `value` for each level of `by`, indexed by
      the levels of `by` -- i.e., the result of df.groupby(by)[value].mean()
    '''
    raise NotImplementedError("You need to write this part!")

In [ ]:
help(load_and_clean)
help(to_feature_matrix)
help(group_means)

**Check `load_and_clean`.**  When it works, this should print two conditions with 150 samples each, and `gene_d` should have a float dtype:

In [ ]:
df2 = load_and_clean('a12pm_measurements.csv')
print(df2['condition'].value_counts())
print('gene_d dtype:', df2['gene_d'].dtype)
print('rows:', len(df2))

**Check `to_feature_matrix`.**  Expected output:

```
X: (250, 4) float64    y: (250,)
labels: ['control' 'treatment']
```

In [ ]:
df2 = load_and_clean('a12pm_measurements.csv')
X2, y2 = to_feature_matrix(df2, ['gene_a','gene_b','gene_c','gene_d'], 'condition')
print('X:', X2.shape, X2.dtype, '   y:', y2.shape)
print('labels:', np.unique(y2))

**Check `group_means`.**  This should match the Section 6 table's `gene_a` column (control ≈ 4.85, treatment ≈ 6.12):

In [ ]:
df2 = load_and_clean('a12pm_measurements.csv')
print(group_means(df2, 'condition', 'gene_a'))

**The real homework** starts after the workshop: do this to one of your own spreadsheets.  `load_and_clean` will need changing — your mess is different from our mess — but the shape of the function is exactly the same.